In [2]:
from pathlib import Path
import pandas as pd

# Set the project folder
PROJECT_DIR = Path.home() / "Desktop" / "FlowCast"

# Load the cleaned datasets
traffic = pd.read_csv(
    PROJECT_DIR / "data" / "processed" / "traffic_cleaned.csv"
)

weather = pd.read_csv(
    PROJECT_DIR / "data" / "processed" / "weather_cleaned.csv"
)

calendar = pd.read_csv(
    PROJECT_DIR / "data" / "processed" / "calendar_cleaned.csv"
)

# Display dataset sizes
print("Traffic dataset:", traffic.shape)
print("Weather dataset:", weather.shape)
print("Calendar dataset:", calendar.shape)

# Display column names
print("\nTraffic columns:")
print(traffic.columns.tolist())

print("\nWeather columns:")
print(weather.columns.tolist())

print("\nCalendar columns:")
print(calendar.columns.tolist())

Traffic dataset: (176701, 20)
Weather dataset: (10872, 8)
Calendar dataset: (151, 6)

Traffic columns:
['road_id', 'road_name', 'latitude', 'longitude', 'weather_station_id', 'date', 'time', 'traffic_volume', 'vehicle_count', 'vehicle_type_dist', 'avg_speed', 'occupancy', 'congestion_level', 'travel_time', 'accident_count', 'signal_timing', 'road_capacity', 'timestamp', 'capacity_30min', 'volume_capacity_ratio']

Weather columns:
['station_id', 'date', 'time', 'weather_condition', 'temperature', 'rainfall', 'visibility', 'timestamp']

Calendar columns:
['date', 'public_holiday', 'holiday_name', 'event_flag', 'event_name', 'roadwork_flag']


In [3]:
# Check date and time formats before merging
print("Traffic date examples:")
print(traffic["date"].head())

print("\nTraffic time examples:")
print(traffic["time"].head())

print("\nWeather station examples:")
print(weather["station_id"].unique()[:10])

print("\nTraffic weather station examples:")
print(traffic["weather_station_id"].unique()[:10])

print("\nCalendar date examples:")
print(calendar["date"].head())

Traffic date examples:
0    2025-01-01
1    2025-01-01
2    2025-01-01
3    2025-01-01
4    2025-01-01
Name: date, dtype: object

Traffic time examples:
0    00:00
1    00:30
2    01:00
3    01:30
4    02:00
Name: time, dtype: object

Weather station examples:
['WS-CENTRAL' 'WS-NORTH' 'WS-SOUTH']

Traffic weather station examples:
['WS-NORTH' 'WS-CENTRAL' 'WS-SOUTH']

Calendar date examples:
0    2025-01-01
1    2025-01-02
2    2025-01-03
3    2025-01-04
4    2025-01-05
Name: date, dtype: object


In [4]:
# Convert traffic date and time 

traffic["date"] = pd.to_datetime(
    traffic["date"].astype(str).str.strip(),
    errors="coerce",
    format="%Y-%m-%d"
)

traffic["time"] = traffic["time"].astype(str).str.strip()

traffic["timestamp"] = pd.to_datetime(
    traffic["date"].dt.strftime("%Y-%m-%d")
    + " "
    + traffic["time"],
    format="%Y-%m-%d %H:%M",
    errors="coerce"
)

# Convert weather date and time
weather["date"] = pd.to_datetime(
    weather["date"].astype(str).str.strip(),
    format="%d/%m/%Y",
    errors="coerce"
)

weather["time"] = weather["time"].astype(str).str.strip()

weather["timestamp"] = pd.to_datetime(
    weather["date"].dt.strftime("%Y-%m-%d")
    + " "
    + weather["time"],
    format="%Y-%m-%d %H:%M",
    errors="coerce"
)

# Convert calendar dates
calendar["date"] = pd.to_datetime(
    calendar["date"].astype(str).str.strip(),
    errors="coerce",
    format="%Y-%m-%d"
).dt.normalize()

print("Traffic missing timestamps:", traffic["timestamp"].isna().sum())
print("Weather missing timestamps:", weather["timestamp"].isna().sum())
print("Calendar missing dates:", calendar["date"].isna().sum())

display(traffic[["date", "time", "timestamp"]].head())
display(weather[["date", "time", "timestamp"]].head())

Traffic missing timestamps: 0
Weather missing timestamps: 0
Calendar missing dates: 0


,date,time,timestamp
0,2025-01-01,00:00,2025-01-01 00:00:00
1,2025-01-01,00:30,2025-01-01 00:30:00
2,2025-01-01,01:00,2025-01-01 01:00:00
3,2025-01-01,01:30,2025-01-01 01:30:00
4,2025-01-01,02:00,2025-01-01 02:00:00


,date,time,timestamp
0,2025-01-01,00:00,2025-01-01 00:00:00
1,2025-01-01,01:00,2025-01-01 01:00:00
2,2025-01-01,02:00,2025-01-01 02:00:00
3,2025-01-01,03:00,2025-01-01 03:00:00
4,2025-01-01,04:00,2025-01-01 04:00:00


In [5]:
print("Traffic date data type:", traffic["date"].dtype)
print("Traffic time data type:", traffic["time"].dtype)

print("\nTraffic date examples:")
print(traffic["date"].head(10).tolist())

print("\nTraffic time examples:")
print(traffic["time"].head(10).tolist())

print("\nRows with missing traffic timestamps:")
display(
    traffic.loc[
        traffic["timestamp"].isna(),
        ["date", "time", "timestamp"]
    ].head(10)
)

Traffic date data type: datetime64[ns]
Traffic time data type: object

Traffic date examples:
[Timestamp('2025-01-01 00:00:00'), Timestamp('2025-01-01 00:00:00'), Timestamp('2025-01-01 00:00:00'), Timestamp('2025-01-01 00:00:00'), Timestamp('2025-01-01 00:00:00'), Timestamp('2025-01-01 00:00:00'), Timestamp('2025-01-01 00:00:00'), Timestamp('2025-01-01 00:00:00'), Timestamp('2025-01-01 00:00:00'), Timestamp('2025-01-01 00:00:00')]

Traffic time examples:
['00:00', '00:30', '01:00', '01:30', '02:00', '02:30', '03:00', '03:30', '04:30', '05:00']

Rows with missing traffic timestamps:


,date,time,timestamp


In [6]:
# Check that timestamps are valid and inspect merge keys

print("Traffic rows:", len(traffic))
print("Weather rows:", len(weather))
print("Calendar rows:", len(calendar))

print("\nTraffic timestamp range:")
print(traffic["timestamp"].min(), "to", traffic["timestamp"].max())

print("\nWeather timestamp range:")
print(weather["timestamp"].min(), "to", weather["timestamp"].max())

print("\nTraffic weather station IDs:")
print(traffic["weather_station_id"].unique())

print("\nWeather station IDs:")
print(weather["station_id"].unique())

print("\nDuplicate traffic timestamps by road:")
print(
    traffic.duplicated(
        subset=["road_id", "timestamp"]
    ).sum()
)

Traffic rows: 176701
Weather rows: 10872
Calendar rows: 151

Traffic timestamp range:
2025-01-01 00:00:00 to 2025-05-31 23:30:00

Weather timestamp range:
2025-01-01 00:00:00 to 2025-05-31 23:00:00

Traffic weather station IDs:
['WS-NORTH' 'WS-CENTRAL' 'WS-SOUTH']

Weather station IDs:
['WS-CENTRAL' 'WS-NORTH' 'WS-SOUTH']

Duplicate traffic timestamps by road:
0


In [7]:
# Step 1: Prepare copies for merging
traffic_merge = traffic.copy()
weather_merge = weather.copy()
calendar_merge = calendar.copy()

# Step 2: Ensure timestamps are datetime values
traffic_merge["timestamp"] = pd.to_datetime(
    traffic_merge["timestamp"], errors="coerce"
)
weather_merge["timestamp"] = pd.to_datetime(
    weather_merge["timestamp"], errors="coerce"
)
calendar_merge["date"] = pd.to_datetime(
    calendar_merge["date"], errors="coerce"
).dt.normalize()

# Remove rows with invalid merge keys
traffic_merge = traffic_merge.dropna(
    subset=["road_id", "timestamp", "weather_station_id"]
).copy()

weather_merge = weather_merge.dropna(
    subset=["station_id", "timestamp"]
).copy()

calendar_merge = calendar_merge.dropna(
    subset=["date"]
).copy()

# Step 3: Ensure weather has one record per station and timestamp
weather_merge = weather_merge.drop_duplicates(
    subset=["station_id", "timestamp"]
)

# Rename station_id to match traffic's weather_station_id
weather_merge = weather_merge.rename(
    columns={"station_id": "weather_station_id"}
)

# Step 4: Match each traffic row with the latest weather
# observation from the same station at or before that time.
traffic_merge = traffic_merge.sort_values("timestamp")
weather_merge = weather_merge.sort_values("timestamp")

merged = pd.merge_asof(
    traffic_merge,
    weather_merge,
    on="timestamp",
    by="weather_station_id",
    direction="backward",
    tolerance=pd.Timedelta("60min"),
    suffixes=("", "_weather")
)

# Step 5: Merge calendar information by date
merged["date_key"] = merged["timestamp"].dt.normalize()

calendar_merge = calendar_merge.drop_duplicates(subset=["date"])

merged = merged.merge(
    calendar_merge,
    left_on="date_key",
    right_on="date",
    how="left",
    suffixes=("", "_calendar"),
    validate="many_to_one"
)

# Remove helper columns
merged = merged.drop(
    columns=["date_key", "date_calendar"],
    errors="ignore"
)

# Step 6: Check the result
print("Original traffic rows:", len(traffic))
print("Merged rows:", len(merged))
print("Merged columns:", len(merged.columns))

print("\nMissing weather condition values:")
print(merged["weather_condition"].isna().sum())

print("\nMissing calendar dates:")
print(merged["public_holiday"].isna().sum())

display(merged.head())

Original traffic rows: 176701
Merged rows: 176701
Merged columns: 31

Missing weather condition values:
0

Missing calendar dates:
0


,road_id,road_name,latitude,longitude,weather_station_id,date,time,traffic_volume,vehicle_count,vehicle_type_dist,...,time_weather,weather_condition,temperature,rainfall,visibility,public_holiday,holiday_name,event_flag,event_name,roadwork_flag
0,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-01,00:00,97.0,97,"{""2W"": 0.39, ""Car"": 0.35, ""LCV"": 0.12, ""HCV"": ...",...,00:00,Clear,12.7,0.0,10000.0,1,New Year's Day,0,NaN,0
1,NL-012,Market St @ Exchange,28.671280,77.258765,WS-SOUTH,2025-01-01,00:00,72.0,72,"{""2W"": 0.43, ""Car"": 0.36, ""LCV"": 0.12, ""HCV"": ...",...,00:00,Clear,7.6,0.0,10000.0,1,New Year's Day,0,NaN,0
2,NL-015,Airport Feeder,28.626508,77.212314,WS-SOUTH,2025-01-01,00:00,115.0,115,"{""2W"": 0.4, ""Car"": 0.35, ""LCV"": 0.16, ""HCV"": 0...",...,00:00,Clear,7.6,0.0,10000.0,1,New Year's Day,0,NaN,0
3,NL-013,Industrial Link A,28.647981,77.161851,WS-NORTH,2025-01-01,00:00,73.0,73,"{""2W"": 0.4, ""Car"": 0.4, ""LCV"": 0.1, ""HCV"": 0.1}",...,00:00,Clear,12.7,0.0,10000.0,1,New Year's Day,0,NaN,0
4,NL-014,Industrial Link B,28.594749,77.253501,WS-CENTRAL,2025-01-01,00:00,119.0,119,"{""2W"": 0.4, ""Car"": 0.33, ""LCV"": 0.16, ""HCV"": 0...",...,00:00,Clear,10.2,0.0,10000.0,1,New Year's Day,0,NaN,0


In [8]:
# Create the processed-data folder
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Save the merged dataset
output_path = PROCESSED_DIR / "flowcast_merged.csv"

merged.to_csv(output_path, index=False)

print("Merged dataset saved successfully!")
print("File location:", output_path)
print("Total rows:", len(merged))
print("Total columns:", len(merged.columns))

Merged dataset saved successfully!
File location: C:\Users\pathi\Desktop\FlowCast\data\processed\flowcast_merged.csv
Total rows: 176701
Total columns: 31


In [9]:
# FlowCast Data Quality Report

import pandas as pd
from pathlib import Path

merged = pd.read_csv(
    PROJECT_DIR / "data" / "processed" / "flowcast_merged.csv"
)

quality_report = pd.DataFrame({
    "Column": merged.columns,
    "Missing Values": merged.isnull().sum().values,
    "Missing Percentage": (
        merged.isnull().mean().values * 100
    ).round(2),
    "Data Type": merged.dtypes.astype(str).values
})

print("Dataset Shape:", merged.shape)
print("Duplicate Rows:", merged.duplicated().sum())

display(quality_report)

report_path = (
    PROJECT_DIR / "data" / "processed" /
    "data_quality_report.csv"
)

quality_report.to_csv(report_path, index=False)

print("Data quality report saved:", report_path)

C:\Users\pathi\AppData\Local\Temp\ipykernel_10244\1332851372.py:6: DtypeWarning: Columns (27,29) have mixed types. Specify dtype option on import or set low_memory=False.
  merged = pd.read_csv(


Dataset Shape: (176701, 31)
Duplicate Rows: 0


,Column,Missing Values,Missing Percentage,Data Type
0,road_id,0,0.00,object
1,road_name,0,0.00,object
2,latitude,0,0.00,float64
3,longitude,0,0.00,float64
4,weather_station_id,0,0.00,object
5,date,0,0.00,object
6,time,0,0.00,object
7,traffic_volume,0,0.00,float64
8,vehicle_count,0,0.00,int64
9,vehicle_type_dist,0,0.00,object


Data quality report saved: C:\Users\pathi\Desktop\FlowCast\data\processed\data_quality_report.csv
